# Lab 1.1 &mdash; From a Stateless Call to an Agent Loop

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Carry state the way LangChain does &mdash; a list of messages you resend
- Let the model choose a tool for real, with <code>bind_tools</code> and <code>tool_calls</code>
- Close the loop, and decide what makes it stop
- Then replace the whole thing with <code>create_agent</code> and compare

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 1 labs work one case: a small tech-support ticket queue.
> What you build in each lab is picked up by the next one.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# A small tech-support ticket queue. Ordinary rules on purpose: the only new thing in these
# five labs is LangChain. Nothing here is real data and nothing leaves this notebook.

TICKETS = {
    "TCK-4001": {"customer": "Priya Nair",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "VPN-513",
                 "text": "Cannot connect since the upgrade. Error VPN-513."},
    "TCK-4002": {"customer": "Rahul Menon",  "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Monthly export finishes but the PDF is blank."},
    "TCK-4003": {"customer": "Anita Sharma", "product": "Reports",    "version": "3.9.1",
                 "severity": "medium", "error_code": None,
                 "text": "It is just slow today. Nothing else to add."},
    "TCK-4004": {"customer": "Vikram Rao",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "SEC-900",
                 "text": "Got a login alert from a country I have never visited."},
    "TCK-4005": {"customer": "Priya Nair",   "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Same blank PDF as my colleague reported."},
}

# The runbook: what support is allowed to do about each error code.
RUNBOOK = {
    "VPN-513": "Certificate pinning changed in 4.2. Have the user clear the local trust store "
               "and re-enrol. Five minutes, no data loss. Support may do this without approval.",
    "APP-002": "Known defect in 3.9.1, fixed in 3.9.2. Advise the upgrade. Do not issue a refund "
               "for this and do not raise a new defect -- link the existing one.",
    "SEC-900": "Possible credential compromise. Escalate to the security desk immediately. "
               "Support must not resolve, close or advise the customer directly.",
}

# Which error codes may an agent resolve on its own, and which need a human?
MUST_ESCALATE = {"SEC-900"}

print(f"{len(TICKETS)} tickets, {len(RUNBOOK)} runbook entries loaded")

## Concept

A model call is a **function**: messages in, one message out. It remembers nothing. Everything an
agent appears to know is something your code put back in front of it.

An agent is that function inside a loop:

| Piece | What it is |
|---|---|
| `llm.bind_tools([...])` | a model allowed to answer with a **tool call** instead of prose |
| `msg.tool_calls` | what it asked for &mdash; name and arguments, already parsed |
| `ToolMessage` | the result you hand back, tied to the request by `tool_call_id` |
| the loop | resend, run what it asked for, resend again &mdash; until you stop |

Three decisions in this lab, and they are the three that decide whether an agent works: what you
resend, what you append, and when you stop.

## Section 1 &mdash; Messages are the state

Two tools and a history. The model sees only what is in that list.

In [ ]:
import json
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

SYSTEM = ("You are a tech support analyst. Answer only from the data the tools give you. "
          "If the runbook says escalate, say so and stop.")

@tool
def lookup_ticket(ref: str) -> str:
    """Return the support ticket for one reference such as 'TCK-4001'.

    Use when you need the customer, product, severity or error code of a specific ticket.
    """
    t = TICKETS.get(ref)
    return json.dumps({"ref": ref, **t}) if t else f"no ticket {ref!r}"

@tool
def runbook_for(error_code: str) -> str:
    """Return what support is allowed to do about one error code, e.g. 'VPN-513'.

    Use after you know why a ticket failed and need to know what to do about it.
    """
    return RUNBOOK.get(error_code, f"no runbook entry for {error_code!r}")

TOOLS = {t.name: t for t in (lookup_ticket, runbook_for)}


def new_history(question: str) -> list:
    """The model is stateless. What does it need in front of it to answer at all?"""
    with_instructions = [SystemMessage(SYSTEM), HumanMessage(question)]
    question_only     = [HumanMessage(question)]
    return BLANK        # TODO: which one, and why does it have to be resent every turn?

In [ ]:
def advance(history: list, ai_msg) -> list:
    """The model asked for tools. Run them, and return the history for the next turn."""
    results = [ToolMessage(content=str(TOOLS[c["name"]].invoke(c["args"])), tool_call_id=c["id"])
               for c in ai_msg.tool_calls]

    results_only         = history + results
    request_and_results  = history + [ai_msg] + results

    return BLANK        # TODO: what must the model see on its next turn?

In [ ]:
# --- Self-check: Section 1   (real tools and real message objects -- no model call)
class _FakeAI:
    """Stands in for the model's reply so this check needs no endpoint."""
    tool_calls = [{"name": "lookup_ticket", "args": {"ref": "TCK-4001"}, "id": "call_1"}]

check("the history the model sees carries your instructions, not just the question",
      lambda: any(isinstance(m, SystemMessage) for m in new_history("what is wrong with TCK-4001?")),
      "nothing is remembered between calls -- the system prompt is resent every turn")
check("a tool result goes back with the request that asked for it",
      lambda: [type(m).__name__ for m in advance([], _FakeAI())] == ["_FakeAI", "ToolMessage"],
      "a ToolMessage with no AIMessage before it is an orphan the model cannot match up")
check("the result carries the real tool output and the call's id",
      lambda: (lambda m: "Priya Nair" in m.content and m.tool_call_id == "call_1")(
              advance([], _FakeAI())[-1]))
score()

## Section 2 &mdash; The loop, and what stops it

Resend, run what it asked for, resend again. The only hard part is the exit.

Three things can end a run, and an agent that checks only the first is the single most common
thing to go wrong in production.

In [ ]:
MAX_STEPS = 6

def should_stop(ai_msg, steps: int, seen: list) -> bool:
    """Three reasons a run should end. Which of them count?"""
    answered      = not ai_msg.tool_calls
    budget_spent  = steps >= MAX_STEPS
    going_in_circles = len(seen) != len(set(seen))

    return BLANK        # TODO: combine the ones that should end the loop

In [ ]:
def run_agent(question: str, model=None):
    """The whole agent: history, model, tools, loop, stop."""
    model = model or get_llm().bind_tools(list(TOOLS.values()))
    history, seen, steps = new_history(question), [], 0
    while True:
        ai = model.invoke(history)
        seen += [f'{c["name"]}({c["args"]})' for c in ai.tool_calls]
        steps += 1
        if should_stop(ai, steps, seen):
            return ai, seen, steps
        history = advance(history, ai)


# --- Self-check: Section 2   (the stop rule alone -- pure function, no model)
class _Ask:  tool_calls = [{"name": "lookup_ticket", "args": {}, "id": "c"}]
class _Done: tool_calls = []

check("a reply with no tool call ends the run",
      lambda: should_stop(_Done(), 1, []) is True)
check("the step budget ends a run that keeps asking",
      lambda: should_stop(_Ask(), MAX_STEPS, ["a", "b"]) is True,
      "without this an agent that never converges never stops")
check("a repeated tool call ends it too",
      lambda: should_stop(_Ask(), 2, ["same", "same"]) is True,
      "the same call twice means it is not learning anything from the results")
check("an ordinary turn in progress does not stop",
      lambda: should_stop(_Ask(), 2, ["a", "b"]) is False)
score()

## Run it for real &mdash; part 1: the model remembers nothing

In [ ]:
if llm_ready():
    print("Q1:", ask("Ticket TCK-4001 is about a VPN client. Reply with just the product name.")[:80])
    print("Q2:", ask("Which ticket did I just ask you about?")[:120])
    print("\n^ the second call has no idea. Nothing was carried over, because nothing carries itself.")

## Run it for real &mdash; part 2: your loop, then the one-liner

`TCK-4004` is the interesting one: the model has to look the ticket up, find `SEC-900`, read the
runbook, and discover it is not allowed to act.

In [ ]:
def show(label, question):
    ai, seen, steps = run_agent(question)
    print(f"=== {label} ===")
    for s in seen:
        print("   called:", s)
    print(f"   steps: {steps}\n   answer: {str(ai.content)[:260]}\n")

if llm_ready():
    guard(lambda: show("your loop", "What should we do about ticket TCK-4004?"))

In [ ]:
if llm_ready():
    from langchain.agents import create_agent
    from langgraph.checkpoint.memory import InMemorySaver

    agent = create_agent(model=get_llm(), tools=list(TOOLS.values()),
                         system_prompt=SYSTEM, checkpointer=InMemorySaver())
    cfg = {"configurable": {"thread_id": "TCK-4004"}}

    out = agent.invoke({"messages": [("user", "What should we do about ticket TCK-4004?")]}, cfg)
    print("create_agent:", out["messages"][-1].content[:260])

    # and because it has a checkpointer, the follow-up needs no context from you
    out = agent.invoke({"messages": [("user", "Which customer was that?")]}, cfg)
    print("\nfollow-up  :", out["messages"][-1].content[:160])

### Read it

1. **The loop is the agent.** Not the model, not the tools &mdash; the loop that resends. You wrote
   about fifteen lines and it is a working agent.
2. **`create_agent` is that loop.** Not a different thing: the same thing, with the message
   plumbing and the stop rule already written. Note the argument is `system_prompt=`, not `prompt=`
   &mdash; `prompt=` raises `TypeError` on this version.
3. **The follow-up question worked.** A `checkpointer` plus a `thread_id` is Module 3's whole
   subject, arriving four hours early because it is one keyword argument.
4. **`TCK-4004` stopped rather than helped.** The runbook said escalate and the agent said so.
   That behaviour came from the data, not from the model being careful &mdash; which is Module 8's
   entire argument.

In [ ]:
score()

## Your turn

1. Set `MAX_STEPS = 1` and run `TCK-4004`. The agent answers from a single lookup, confidently and
   wrongly. A budget too tight is its own failure mode.
2. Ask about `TCK-4003`, which has no error code. Watch what the agent does when a tool returns
   nothing useful &mdash; that is the failure Module 4 is about.